In [2]:
%pip install -Uq langchain-core langchain-community langchain-openai langsmith python-dotenv

In [4]:
# ===== Step 0: install (Colab / notebook) =====
# %pip install -U langsmith langchain-core langchain-community langchain-openai python-dotenv

import os
from google.colab import userdata

# ===== Step 1: keys & LangSmith tracing env =====
# OpenAI key
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# LangSmith key (請確認在 Colab 的 Secrets 裡是用這個名稱存的)
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")

# Enable tracing, 設為 true 之後，LangChain 在 runtime 會自動啟用 LangSmith client
os.environ["LANGSMITH_TRACING"] = "true"

# Optional: 指定 project，否則會進 default
os.environ["LANGSMITH_PROJECT"] = "demo-langsmith-tracing-on-colab"

# Optional: 只有在自架 LangSmith / 特定區域時才需要手動指定
# os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"


# ===== Step 2: your LangChain app (will be traced automatically) =====
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Please respond only based on the given context."),
        ("user", "Question: {question}\nContext: {context}"),
    ]
)

# 模型名稱請依你帳號可用的 OpenAI model 為準
model = ChatOpenAI(model="gpt-4o-mini")
chain = prompt | model | StrOutputParser()

question = "Can you summarize this morning's meetings?"
context = "During this morning's meeting, we decided to start new amazing project that will make a lot of money."

result = chain.invoke(
    {"question": question, "context": context},
    config={
        # 這些會讓你在 LangSmith UI 更好搜、更好分群
        "run_name": "meeting_summarizer",
        "tags": ["demo", "meeting"],
        "metadata": {"env": "colab", "example": True},
    },
)

print(result)


# ===== Optional: 精準控制「這一次要不要 trace」(不改環境變數) =====
# import langsmith as ls
# with ls.tracing_context(enabled=True, project_name="demo-langsmith-tracing"):
#     chain.invoke({"question": question, "context": context}, config={"run_name": "meeting_summarizer_ctx"})

This morning's meeting focused on launching a new project that is expected to be highly profitable.
